# Single-frame (2D CNN) force estimator - exploration and test-set evaluation

2D counterpart of `Model/explore.ipynb`: the model sees only each window's label (centre) frame, so comparing its numbers with the 3D model's shows what the temporal context adds. Reads `Model2D/checkpoints/metrics_history.csv` (written every epoch by `train.py`) and `Model2D/checkpoints/best_model.pt` (the checkpoint with the best validation MAE), then evaluates that checkpoint on the held-out test session. Run this after `mainModel.py` has finished a training run.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import dataset
import model as model_module
import train as train_module

config = dataset.load_config()
model_config = config[dataset.CONFIG_SECTION]
checkpoint_dir = os.path.join(dataset.REPO_ROOT, model_config["training"]["checkpoint_dir"])
metrics_path = os.path.join(checkpoint_dir, model_config["training"]["metrics_history_file"])
checkpoint_path = os.path.join(checkpoint_dir, "best_model.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"checkpoint_dir={checkpoint_dir}\ndevice={device}")

## Training curves

Train/val Huber loss (normalized units) and validation MAE (newtons) over epochs.

In [ ]:
history = pd.read_csv(metrics_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("Huber loss (normalized)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title("Train/val loss")

axes[1].plot(history["epoch"], history["val_mae_n"], color="tab:red")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val MAE (N)"); axes[1].grid(alpha=0.3)
axes[1].set_title("Validation MAE")
fig.tight_layout()

best_epoch = history.loc[history["val_mae_n"].idxmin()]
print(f"Best epoch: {int(best_epoch['epoch'])}  val_mae_n={best_epoch['val_mae_n']:.3f}  val_r2={best_epoch['val_r2']:.3f}")

## Load the best checkpoint

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
print("Checkpoint metadata:")
for key, value in checkpoint["metadata"].items():
    print(f"  {key}: {value}")

net = model_module.build_model(mean_normalized_label=0.0).to(device)
net.load_state_dict(checkpoint["model_state_dict"])
net.eval()

clamp_sign = checkpoint["metadata"]["prediction_clamp_sign"]
force_norm_scale_n = checkpoint["metadata"]["force_norm_scale_n"]
label_offset = checkpoint["metadata"]["label_frame_offset"]
autocast_dtype, _ = train_module.select_precision(device)
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}, val_mae_n={checkpoint['val_mae_n']:.3f}")

## Sample frames (sanity check)

De-normalized color and relative-depth channel of a few training-split samples (augmentation on), with the true label. Confirms color order, crop placement, and that a geometric augmentation (if it fired) hits color and depth identically.

In [ ]:
train_dataset = dataset.build_dataset("train", config)

num_samples = 4
fig, axes = plt.subplots(num_samples, 2, figsize=(6, 3 * num_samples))
sample_indices = np.linspace(0, len(train_dataset) - 1, num_samples, dtype=int)
for row, sample_index in enumerate(sample_indices):
    item = train_dataset[int(sample_index)]
    image = item["image"]
    color_frame = image[:3].permute(1, 2, 0).numpy() * dataset.RGB_STD + dataset.RGB_MEAN
    axes[row, 0].imshow(np.clip(color_frame, 0, 1))
    axes[row, 0].set_title(f"Fz={item['label_n'].item():.1f}N  frame {item['label_frame']}"); axes[row, 0].axis("off")
    axes[row, 1].imshow(image[3].numpy(), cmap="coolwarm", vmin=-1, vmax=1)
    axes[row, 1].set_title("relative depth"); axes[row, 1].axis("off")
fig.suptitle("Sample training frames: natural skin (RGB order), depth blob on the finger")
fig.tight_layout()

## Test-set evaluation

MAE, RMSE, R² in newtons on the held-out test sessions, evaluated densely (eval_stride=1). With `TEST_APPLY_WINDOW_FILTERS = True` only windows passing the training rules (max_span_s / max_force_change_n) are scored; set it to `False` to score every window start in every test session. The scatter, time-series and per-bin cells below use the same windows.

In [ ]:
TEST_APPLY_WINDOW_FILTERS = False   # False = score every window start, ignoring max_span_s / max_force_change_n

test_dataset, test_loader = dataset.build_dataloader("test", config, shuffle=False)
if not TEST_APPLY_WINDOW_FILTERS:
    test_dataset.windows = [(session_index, start)
                            for session_index, session in enumerate(test_dataset.sessions)
                            for start in range(session.num_frames - test_dataset.window_length + 1)]
loss_fn = torch.nn.HuberLoss(delta=model_config["training"]["loss"]["huber_delta_n"] / force_norm_scale_n)
test_loss, test_mae_n, test_rmse_n, test_r2 = train_module.run_validation(
    net, test_loader, device, autocast_dtype, loss_fn, force_norm_scale_n, clamp_sign)
filter_note = "filtered" if TEST_APPLY_WINDOW_FILTERS else "unfiltered"
print(f"Test set ({len(test_dataset)} {filter_note} windows): MAE={test_mae_n:.3f} N  RMSE={test_rmse_n:.3f} N  R2={test_r2:.3f}")

## Predicted vs. true force (scatter)

In [ ]:
preds_n, trues_n, start_frames = [], [], []
net.eval()
with torch.inference_mode():
    for batch in test_loader:
        image = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=autocast_dtype is not None):
            prediction = net(image).squeeze(1).float()
        prediction = train_module._clamp_predictions(prediction, clamp_sign)
        preds_n.append(prediction.cpu().numpy() * force_norm_scale_n)
        trues_n.append(batch["label_n"].numpy())
        start_frames.append(batch["start_frame"].numpy())
preds_n = np.concatenate(preds_n)
trues_n = np.concatenate(trues_n)
start_frames = np.concatenate(start_frames)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(trues_n, preds_n, s=4, alpha=0.3)
lims = [min(trues_n.min(), preds_n.min()), max(trues_n.max(), preds_n.max())]
ax.plot(lims, lims, "k--", label="identity")
ax.set_xlabel("true Fz (N)"); ax.set_ylabel("predicted Fz (N)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Test set: predicted vs. true force")
fig.tight_layout()

## Predicted vs. true force over time (full test trial)

Plots the first test session, evaluated at every frame (eval_stride=1), so this reconstructs a dense predicted-force curve over the whole trial - the view that matters for path planning, and where lag or noise becomes obvious.

In [ ]:
test_session = test_dataset.sessions[0]
label_frame_indices = start_frames + label_offset
elapsed_s = (test_session.frame_ts_ns[label_frame_indices] - test_session.frame_ts_ns[0]) / 1e9
order = np.argsort(elapsed_s)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(elapsed_s[order], trues_n[order], label="true (filtered Fz)", linewidth=1)
ax.plot(elapsed_s[order], preds_n[order], label="predicted", linewidth=1, alpha=0.8)
ax.set_xlabel("time (s)"); ax.set_ylabel("Fz (N)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"Test trial ({test_session.name}): predicted vs. true force over time")
fig.tight_layout()

## Error by force bin

MAE within each 2N true-force bin - shows whether the model regresses toward the mean and underestimates high forces.

In [ ]:
bin_width_n = 2.0
bin_edges = np.arange(np.floor(trues_n.min() / bin_width_n) * bin_width_n,
                       np.ceil(trues_n.max() / bin_width_n) * bin_width_n + bin_width_n, bin_width_n)
bin_index = np.digitize(trues_n, bin_edges)

bin_centers, bin_maes, bin_counts = [], [], []
for b in sorted(set(bin_index)):
    mask = bin_index == b
    if mask.sum() == 0:
        continue
    low = bin_edges[b - 1] if 0 < b <= len(bin_edges) else bin_edges[0]
    bin_centers.append(low + bin_width_n / 2)
    bin_maes.append(np.mean(np.abs(preds_n[mask] - trues_n[mask])))
    bin_counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(bin_centers, bin_maes, width=bin_width_n * 0.9, color="tab:blue")
for bar, count in zip(bars, bin_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count),
            ha="center", va="bottom", fontsize=8)
ax.set_xlabel("true Fz bin center (N)"); ax.set_ylabel("MAE (N)")
ax.set_title("MAE per 2N force bin (bar label = sample count)")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

## Single session, start to finish

Runs one session (from any split) through the model at every possible window start (stride 1), ignoring the training-time window filters, so the prediction covers the whole recording. Change `SESSION_SPLIT` / `SESSION_INDEX` to pick the session, e.g. `"test", 1` = the second entry of `test_sessions` in config.yaml. Set `APPLY_WINDOW_FILTERS = True` to only predict on windows that pass the span/force-change rules. Lines are broken at timestamp gaps larger than `max_span_s`.

In [ ]:
SESSION_SPLIT = "test"          # "train", "val" or "test"
SESSION_INDEX = 0               # index into config.yaml's <split>_sessions list
APPLY_WINDOW_FILTERS = False    # True = only windows passing max_span_s / max_force_change_n

import copy
from torch.utils.data import DataLoader

session_name = model_config["splits"][f"{SESSION_SPLIT}_sessions"][SESSION_INDEX]
single_config = copy.deepcopy(model_config)
single_config["splits"]["test_sessions"] = [session_name]
# Built as the "test" split so no augmentation / temporal jitter is applied, even for a training session.
single_dataset = dataset.ForceFrameDataset(single_config, "test")
single_session = single_dataset.sessions[0]
if not APPLY_WINDOW_FILTERS:
    single_dataset.windows = [(0, start) for start in range(single_session.num_frames - single_dataset.window_length + 1)]
single_loader = DataLoader(single_dataset, batch_size=model_config["loader"]["batch_size"], shuffle=False,
                           num_workers=model_config["loader"]["num_workers"], pin_memory=True)

session_preds_n, session_starts = [], []
with torch.inference_mode():
    for batch in single_loader:
        image = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=autocast_dtype is not None):
            prediction = net(image).squeeze(1).float()
        prediction = train_module._clamp_predictions(prediction, clamp_sign)
        session_preds_n.append(prediction.cpu().numpy() * force_norm_scale_n)
        session_starts.append(batch["start_frame"].numpy())
session_preds_n = np.concatenate(session_preds_n)
session_label_frames = np.concatenate(session_starts) + label_offset
session_trues_n = single_session.frame_fz_n[session_label_frames]

max_gap_s = model_config["window"]["max_span_s"]
def _break_at_gaps(t_s, values):
    """Inserts NaN between samples more than max_gap_s apart so matplotlib doesn't draw straight lines across gaps."""
    gap_idx = np.where(np.diff(t_s) > max_gap_s)[0] + 1
    return np.insert(t_s, gap_idx, np.nan), np.insert(values.astype(float), gap_idx, np.nan)

frame_t_s = (single_session.frame_ts_ns - single_session.frame_ts_ns[0]) / 1e9
pred_t_s = frame_t_s[session_label_frames]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(*_break_at_gaps(frame_t_s, single_session.frame_fz_n), label="true (filtered Fz)", linewidth=1)
ax.plot(*_break_at_gaps(pred_t_s, session_preds_n), label="predicted", linewidth=1, alpha=0.8)
ax.set_xlabel("time (s)"); ax.set_ylabel("Fz (N)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"{SESSION_SPLIT}[{SESSION_INDEX}] ({session_name}): predicted vs. true force over time")
fig.tight_layout()

session_err = session_preds_n - session_trues_n
print(f"{session_name}: {len(session_preds_n)} windows  MAE={np.abs(session_err).mean():.3f} N  "
      f"RMSE={np.sqrt((session_err ** 2).mean()):.3f} N")

## Single session: with vs. without depth

Runs the same session as above twice through the same checkpoint: once with the normal RGB+depth image, once with the depth channel zeroed (`image[:, 3] = 0`, i.e. "every pixel at the reference depth", the neutral value of the relative-depth channel). This is an input ablation of the depth-trained model, not a separately trained RGB-only model - it shows how much the trained network relies on depth, not how well an RGB-only network could do. Uses `SESSION_SPLIT` / `SESSION_INDEX` / `APPLY_WINDOW_FILTERS` and `single_loader` from the cell above.

In [ ]:
def _predict_session(loader, zero_depth):
    """Runs net over loader; with zero_depth=True the relative-depth channel is set to 0. Returns predictions in N."""
    preds = []
    with torch.inference_mode():
        for batch in loader:
            image = batch["image"].to(device, non_blocking=True)
            if zero_depth:
                image[:, model_module.DEPTH_CHANNEL_INDEX] = 0.0
            with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=autocast_dtype is not None):
                prediction = net(image).squeeze(1).float()
            prediction = train_module._clamp_predictions(prediction, clamp_sign)
            preds.append(prediction.cpu().numpy() * force_norm_scale_n)
    return np.concatenate(preds)

def _stats(preds, trues):
    err = preds - trues
    ss_res = (err ** 2).sum()
    ss_tot = ((trues - trues.mean()) ** 2).sum()
    return {
        "MAE (N)": np.abs(err).mean(),
        "RMSE (N)": np.sqrt((err ** 2).mean()),
        "R2": 1.0 - ss_res / ss_tot,
        "Pearson r": np.corrcoef(preds, trues)[0, 1],
        "bias (N)": err.mean(),
        "median |err| (N)": np.median(np.abs(err)),
        "P95 |err| (N)": np.percentile(np.abs(err), 95),
        "max |err| (N)": np.abs(err).max(),
    }

ablation_preds = {
    "RGB + depth": _predict_session(single_loader, zero_depth=False),
    "RGB only (depth = 0)": _predict_session(single_loader, zero_depth=True),
}

stats_table = pd.DataFrame({name: _stats(preds, session_trues_n) for name, preds in ablation_preds.items()})
stats_table["difference (no depth - depth)"] = stats_table["RGB only (depth = 0)"] - stats_table["RGB + depth"]
print(f"{session_name}: {len(session_trues_n)} windows")
display(stats_table.round(3))

# Contact vs. no-contact split: where does removing depth hurt?
contact_threshold_n = 0.5
contact_mask = np.abs(session_trues_n) >= contact_threshold_n
split_rows = {}
for name, preds in ablation_preds.items():
    abs_err = np.abs(preds - session_trues_n)
    split_rows[name] = {f"MAE |Fz|<{contact_threshold_n}N (N)": abs_err[~contact_mask].mean() if (~contact_mask).any() else np.nan,
                        f"MAE |Fz|>={contact_threshold_n}N (N)": abs_err[contact_mask].mean() if contact_mask.any() else np.nan}
split_table = pd.DataFrame(split_rows)
split_table["difference (no depth - depth)"] = split_table["RGB only (depth = 0)"] - split_table["RGB + depth"]
print(f"no-contact windows: {(~contact_mask).sum()}   contact windows: {contact_mask.sum()}")
display(split_table.round(3))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(*_break_at_gaps(frame_t_s, single_session.frame_fz_n), label="true (filtered Fz)", linewidth=1, color="black")
for (name, preds), color in zip(ablation_preds.items(), ["tab:orange", "tab:green"]):
    axes[0].plot(*_break_at_gaps(pred_t_s, preds), label=name, linewidth=1, alpha=0.8, color=color)
    axes[1].plot(*_break_at_gaps(pred_t_s, preds - session_trues_n), label=name, linewidth=1, alpha=0.8, color=color)
axes[0].set_ylabel("Fz (N)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title(f"{SESSION_SPLIT}[{SESSION_INDEX}] ({session_name}): with vs. without depth")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("prediction error (N)"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()

# Per-force-bin MAE, both variants side by side
bin_width_n = 2.0
ablation_bin_edges = np.arange(np.floor(session_trues_n.min() / bin_width_n) * bin_width_n,
                               np.ceil(session_trues_n.max() / bin_width_n) * bin_width_n + bin_width_n, bin_width_n)
ablation_bin_index = np.clip(np.digitize(session_trues_n, ablation_bin_edges) - 1, 0, len(ablation_bin_edges) - 2)
occupied_bins = np.unique(ablation_bin_index)
bin_centers_ab = ablation_bin_edges[occupied_bins] + bin_width_n / 2
fig, ax = plt.subplots(figsize=(10, 4))
bar_width = bin_width_n * 0.4
for offset, ((name, preds), color) in zip([-bar_width / 2, bar_width / 2], zip(ablation_preds.items(), ["tab:orange", "tab:green"])):
    maes = [np.abs(preds - session_trues_n)[ablation_bin_index == b].mean() for b in occupied_bins]
    ax.bar(bin_centers_ab + offset, maes, width=bar_width, label=name, color=color)
counts = [(ablation_bin_index == b).sum() for b in occupied_bins]
for center, count in zip(bin_centers_ab, counts):
    ax.text(center, ax.get_ylim()[1] * 0.97, str(count), ha="center", va="top", fontsize=8)
ax.set_xlabel("true Fz bin center (N)"); ax.set_ylabel("MAE (N)"); ax.legend(); ax.grid(alpha=0.3, axis="y")
ax.set_title("MAE per 2N force bin, with vs. without depth (top label = sample count)")
fig.tight_layout()